# Pipeline de traitement des donnees OpenFoodFacts avec Polars (Par Echantillon)

Ce notebook presente une methode de lecture et de traitement **par echantillon (batch)** du gros fichier CSV de OpenFoodFacts (~12 Go).

Il produit **deux fichiers CSV** :
1. `produits_sans_nutriscore.csv` – Produits francais **SANS** Nutri-Score, tres fournis en nutriments *(cibles de prediction)*
2. `produits_avec_nutriscore.csv` – Produits francais **AVEC** Nutri-Score etabli et toutes les features nutritionnelles presentes *(donnees d'entrainement du Random Forest)*

In [1]:
import polars as pl
import os

## Configuration
Parametres communs aux deux pipelines.

In [2]:
FILE_PATH = "data_brut.csv"
BATCH_SIZE = 100_000

# Features nutritionnelles requises pour le Random Forest
FEATURES_RF = [
    'energy_100g',
    'sugars_100g',
    'saturated-fat_100g',
    'salt_100g',
    'fiber_100g',
    'proteins_100g',
    'fruits-vegetables-legumes_100g',
]

COLONNES_NUTRITIONNELLES = '^.*_100g$'

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Fichier introuvable : {FILE_PATH}")
print(f"Fichier source : {FILE_PATH}")

Fichier source : data_brut.csv


## Pipeline 1 — Produits francais SANS Nutri-Score
*(cibles de prediction pour le Random Forest)*

Criteres de selection :
- Vendu en France (`countries_en` contient `france`)
- `nutriscore_score` est **null** (pas encore note)
- Au moins **15 colonnes non nulles** (produit suffisamment documente)

In [3]:
destination_sans = "produits_sans_nutriscore.csv"

reader = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

batches_sans = []
i = 1
print("Debut du Pipeline 1 (sans Nutri-Score)...\n")

while True:
    batches = reader.next_batches(1)
    if not batches:
        break
    df_batch = batches[0]

    df_batch = (
        df_batch
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .select([
            pl.col('code'),
            pl.col('product_name'),
            pl.col('nova_group'),
            pl.col('additives_n'),
            pl.col(COLONNES_NUTRITIONNELLES),
            pl.col('nutriscore_score'),
            pl.col('nutriscore_grade'),
            pl.col('environmental_score_score'),
            pl.col('environmental_score_grade')
        ])
        .filter(pl.col('nutriscore_score').is_null())
        .with_columns(
            pl.sum_horizontal(pl.all().is_not_null()).alias('donnees_fournies_count')
        )
        .filter(pl.col('donnees_fournies_count') >= 15)
    )

    batches_sans.append(df_batch)
    print(f"  Echantillon {i} : {len(df_batch)} produits conserves")
    i += 1

df_sans = pl.concat(batches_sans).sort('donnees_fournies_count', descending=True)
df_sans.write_csv(destination_sans)
print(f"\nFichier sauvegarde : {destination_sans}")
print(f"Dimensions : {df_sans.shape}")
df_sans.head()

Debut du Pipeline 1 (sans Nutri-Score)...


C:\Users\Gambey\AppData\Local\Temp\ipykernel_6612\4185333121.py:3: DeprecationWarning: `read_csv_batched` is deprecated; use `scan_csv().collect_batches()` instead.
  reader = pl.read_csv_batched(



  Echantillon 1 : 173 produits conserves
  Echantillon 2 : 176 produits conserves
  Echantillon 3 : 194 produits conserves
  Echantillon 4 : 409 produits conserves
  Echantillon 5 : 239 produits conserves
  Echantillon 6 : 286 produits conserves
  Echantillon 7 : 212 produits conserves


KeyboardInterrupt: 

## Pipeline 2 — Produits francais AVEC Nutri-Score (donnees d'entrainement)
*(reference pour le Random Forest)*

Criteres de selection :
- Vendu en France (`countries_en` contient `france`)
- `nutriscore_grade` est **une lettre valide** (a / b / c / d / e)
- Toutes les **features nutritionnelles cles du Random Forest** sont presentes (pas de null)

In [ ]:
destination_avec = "produits_avec_nutriscore.csv"

GRADES_VALIDES = ['a', 'b', 'c', 'd', 'e']
batches_avec = []
print("Debut du Pipeline 2 (avec Nutri-Score)...\n")

colonnes_base = [
    'countries_en',
    'code',
    'product_name',
    'nova_group',
    'additives_n',
    'nutriscore_score',
    'nutriscore_grade',
    'environmental_score_score',
    'environmental_score_grade'
]

# ---------- Pass 1 : compter les classes disponibles ----------
reader2_count = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

compteurs = {g: 0 for g in GRADES_VALIDES}

while True:
    batches_count = reader2_count.next_batches(1)
    if not batches_count:
        break

    df_count = batches_count[0]
    colonnes_requises = ['countries_en', 'nutriscore_grade'] + FEATURES_RF
    if any(c not in df_count.columns for c in colonnes_requises):
        continue

    dist = (
        df_count
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .with_columns(pl.col('nutriscore_grade').str.to_lowercase().alias('nutriscore_grade'))
        .filter(pl.col('nutriscore_grade').is_in(GRADES_VALIDES))
        .filter(pl.all_horizontal([pl.col(f).is_not_null() for f in FEATURES_RF]))
        .group_by('nutriscore_grade')
        .len()
    )

    for row in dist.iter_rows(named=True):
        compteurs[row['nutriscore_grade']] += row['len']

TARGET_PAR_GRADE = min(compteurs.values()) if all(v > 0 for v in compteurs.values()) else 0
print(f"Comptes initiaux par grade : {compteurs}")
print(f"Cible equilibree par grade : {TARGET_PAR_GRADE}\n")

# ---------- Pass 2 : extraire de facon equilibree ----------
reader2 = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

quota_restant = {g: TARGET_PAR_GRADE for g in GRADES_VALIDES}

def equilibrer_batch(df: pl.DataFrame) -> pl.DataFrame:
    if TARGET_PAR_GRADE == 0:
        return df.head(0)

    df = (
        df
        .with_columns(pl.col('nutriscore_grade').str.to_lowercase().alias('nutriscore_grade'))
        .filter(pl.col('nutriscore_grade').is_in(GRADES_VALIDES))
        .filter(pl.all_horizontal([pl.col(f).is_not_null() for f in FEATURES_RF]))
    )

    morceaux = []
    for g in GRADES_VALIDES:
        reste = quota_restant[g]
        if reste <= 0:
            continue

        part = df.filter(pl.col('nutriscore_grade') == g)
        if len(part) == 0:
            continue

        n = min(len(part), reste)
        part = part.sample(n=n, seed=42, shuffle=True) if len(part) > n else part.head(n)
        quota_restant[g] -= n
        morceaux.append(part)

    return pl.concat(morceaux) if morceaux else df.head(0)

j = 1
while True:
    if all(v == 0 for v in quota_restant.values()):
        print("Quotas atteints pour tous les grades, arret anticipe.")
        break

    batches = reader2.next_batches(1)
    if not batches:
        break

    df_batch = batches[0]

    colonnes_manquantes_base = [c for c in colonnes_base if c not in df_batch.columns]
    colonnes_manquantes_rf = [f for f in FEATURES_RF if f not in df_batch.columns]
    colonnes_manquantes = colonnes_manquantes_base + colonnes_manquantes_rf

    if colonnes_manquantes:
        print(f"  Echantillon {j} ignore (colonnes manquantes: {colonnes_manquantes})")
        j += 1
        continue

    df_batch = (
        df_batch
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .select([
            pl.col('code'),
            pl.col('product_name'),
            pl.col('nova_group'),
            pl.col('additives_n'),
            pl.col(COLONNES_NUTRITIONNELLES),
            pl.col('nutriscore_score'),
            pl.col('nutriscore_grade'),
            pl.col('environmental_score_score'),
            pl.col('environmental_score_grade')
        ])
        .pipe(equilibrer_batch)
        # Nutri-score valide (a/b/c/d/e)
        .filter(
            pl.col('nutriscore_grade').str.to_lowercase().is_in(GRADES_VALIDES)
        )
        # Toutes les features du RF doivent etre presentes
        .filter(
            pl.all_horizontal([pl.col(f).is_not_null() for f in FEATURES_RF])
        )
    )

    batches_avec.append(df_batch)
    print(f"  Echantillon {j} : {len(df_batch)} produits conserves")
    j += 1

if batches_avec:
    df_avec = pl.concat(batches_avec)
else:
    df_avec = pl.DataFrame()

df_avec.write_csv(destination_avec)
print(f"\nFichier sauvegarde : {destination_avec}")
print(f"Dimensions : {df_avec.shape}")
print("\nDistribution des grades :")
if "nutriscore_grade" in df_avec.columns:
    print(df_avec['nutriscore_grade'].value_counts())
df_avec.head()

C:\Users\Gambey\AppData\Local\Temp\ipykernel_22488\3184031282.py:3: DeprecationWarning: `read_csv_batched` is deprecated; use `scan_csv().collect_batches()` instead.
  reader2 = pl.read_csv_batched(


Debut du Pipeline 2 (avec Nutri-Score)...

  Echantillon 1 : 383 produits conserves
  Echantillon 2 : 578 produits conserves
  Echantillon 3 : 513 produits conserves
  Echantillon 4 : 716 produits conserves
  Echantillon 5 : 619 produits conserves
  Echantillon 6 : 561 produits conserves
  Echantillon 7 : 342 produits conserves
  Echantillon 8 : 386 produits conserves
  Echantillon 9 : 313 produits conserves
  Echantillon 10 : 274 produits conserves
  Echantillon 11 : 259 produits conserves
  Echantillon 12 : 470 produits conserves
  Echantillon 13 : 3624 produits conserves
  Echantillon 14 : 607 produits conserves
  Echantillon 15 : 440 produits conserves
  Echantillon 16 : 2508 produits conserves
  Echantillon 17 : 12816 produits conserves
  Echantillon 18 : 16767 produits conserves
  Echantillon 19 : 8567 produits conserves
  Echantillon 20 : 11966 produits conserves
  Echantillon 21 : 8471 produits conserves
  Echantillon 22 : 4858 produits conserves
  Echantillon 23 : 1653 produit

code,product_name,nova_group,additives_n,energy-kj_100g,energy-kcal_100g,energy_100g,energy-from-fat_100g,fat_100g,saturated-fat_100g,butyric-acid_100g,caproic-acid_100g,caprylic-acid_100g,capric-acid_100g,lauric-acid_100g,myristic-acid_100g,palmitic-acid_100g,stearic-acid_100g,arachidic-acid_100g,behenic-acid_100g,lignoceric-acid_100g,cerotic-acid_100g,montanic-acid_100g,melissic-acid_100g,unsaturated-fat_100g,monounsaturated-fat_100g,omega-9-fat_100g,polyunsaturated-fat_100g,omega-3-fat_100g,omega-6-fat_100g,alpha-linolenic-acid_100g,eicosapentaenoic-acid_100g,docosahexaenoic-acid_100g,linoleic-acid_100g,arachidonic-acid_100g,gamma-linolenic-acid_100g,dihomo-gamma-linolenic-acid_100g,…,chloride_100g,calcium_100g,phosphorus_100g,iron_100g,magnesium_100g,zinc_100g,copper_100g,manganese_100g,fluoride_100g,selenium_100g,chromium_100g,molybdenum_100g,iodine_100g,caffeine_100g,taurine_100g,methylsulfonylmethane_100g,ph_100g,fruits-vegetables-legumes_100g,collagen-meat-protein-ratio_100g,cocoa_100g,chlorophyl_100g,carbon-footprint_100g,glycemic-index_100g,water-hardness_100g,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,nutriscore_score,nutriscore_grade,environmental_score_score,environmental_score_grade
i64,str,i64,i64,f64,f64,f64,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str,i64,str,str,str,str,str,str,…,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,i64,str,str,str,f64,str,str,str,i64,str,str,i64,f64,str,str,str,str,str,str,f64,i64,str,i64,str
112302621,"""Potiron et vermicelles""",4,1,109.6,26.0,109.6,null,0.4,0.2,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,28.78125,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3,"""c""",83,"""a"""
112302650,"""Délice de potiron châtaigne""",4,1,143.0,34.2,143.0,null,0.8,0.4,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,31.666667,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3,"""c""",83,"""a"""
112302656,"""Lié big purSoup mouliné de 10 …",4,0,150.0,35.8,150.0,null,1.2,0.4,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,24.75,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3,"""c""",107,"""a-plus"""
112302660,"""Soupe potiron et kiri""",4,4,161.0,38.3,161.0,null,1.5,1.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,34.3125,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3,"""c""",78,"""a"""
112302676,"""Velouté de légumes du soleil""",3,0,147.5,35.0,147.5,null,1.7,0.3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,29.30125,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3,"""c""",98,"""a-plus"""


: 